# BRCA Spatial Transcriptomics — Notebook 2
## Cell-type deconvolution with DestVI

Each Visium spot is ~1–10 cells, so its expression is a **mixture** of cell types. This
notebook estimates, for every spot, *what fraction of it is each cell type* — tumour,
T-cells, fibroblasts, and so on. That's **deconvolution**.

We use **DestVI** (from `scvi-tools`), which is a variational autoencoder — the same model
family as the Mechano-VAE, so the idea should feel familiar:

1. A first VAE (**CondSCVI**) learns, from an annotated **reference** single-cell dataset,
   a latent description of how each cell type expresses genes.
2. A second model (**DestVI**) then explains each spatial spot as a *combination* of those
   learned cell-type signatures, and reads off the mixing weights — the **proportions**.

Pipeline:
1. Rebuild the spatial data with **full-gene raw counts** (Notebook 1 only kept 2,000 HVGs)
2. Load + subsample the **reference atlas** (Wu et al. 2021 breast cancer)
3. Match genes between the two
4. Train **CondSCVI** on the reference
5. Train **DestVI** on the spots → per-spot cell-type proportions
6. Map cell types onto the tissue, and onto the Notebook-1 domains
7. Save for Notebook 3

> Run on **Colab with a T4 GPU**. DestVI is the heaviest step in the project.

## 0 · Setup

In [ ]:
!pip install -q scanpy squidpy scvi-tools

import warnings; warnings.filterwarnings("ignore")
import os, glob, numpy as np, pandas as pd
import scanpy as sc, squidpy as sq
import scvi
from scvi.model import CondSCVI, DestVI
import matplotlib.pyplot as plt

scvi.settings.seed = 0
print("scanpy", sc.__version__, "| scvi-tools", scvi.__version__)

In [ ]:
# same WORKDIR you used in Notebook 1 (local Colab disk, or your Drive path)
WORKDIR = "/content/brca_spatial"
os.makedirs(WORKDIR, exist_ok=True)
DOMAINS_H5AD = os.path.join(WORKDIR, "brca_visium_domains.h5ad")  # output of Notebook 1

## 1 · Spatial data with full-gene raw counts

DestVI needs **raw counts over many genes**. Notebook 1 subset to 2,000 HVGs and scaled the
matrix, so its `counts` layer is too narrow. We fix that here without re-running Notebook 1:

- reload the **raw** Visium slide (all genes, raw counts),
- keep only the spots that **passed QC** in Notebook 1 (match by barcode),
- copy over the **domain** labels from Notebook 1.

The result, `st_adata`, has raw counts in `.X`, every gene, and the domain labels attached.

In [ ]:
# domains + QC-passed spot list from Notebook 1
nb1 = sc.read_h5ad(DOMAINS_H5AD)
keep_spots = nb1.obs_names

# fresh raw slide (full genes, raw counts)
st_adata = sc.datasets.visium_sge(sample_id="V1_Breast_Cancer_Block_A_Section_1")
st_adata.var_names_make_unique()

# align to the QC-passed spots, then carry the domain labels across (barcode-matched)
st_adata = st_adata[keep_spots].copy()
st_adata.obs["domain"] = nb1.obs.loc[st_adata.obs_names, "domain"].values

print("spatial:", st_adata.shape, "| raw counts in .X:", st_adata.X.max() == int(st_adata.X.max()))
print(st_adata.obs["domain"].value_counts())

## 2 · Reference single-cell atlas (Wu et al. 2021)

Deconvolution needs a labelled reference: pure single cells with known cell types. We use
the **Wu et al. 2021** breast-cancer atlas (~100k cells, GEO `GSE176078`), the standard
reference for the breast tumour microenvironment. Its `celltype_major` annotation has ~9
broad types (Cancer Epithelial, T-cells, B-cells, Myeloid, Endothelial, CAFs, PVL, Normal
Epithelial, Plasmablasts).

> **Reference contract** — any annotated reference works here as long as it ends up as an
> AnnData with **raw counts in `.X`** and a **cell-type column in `.obs`**. If you already
> have such a file, set `REF_H5AD` to its path and skip the download. (An alternative source
> is the `cellxgene-census` API, which serves breast datasets with standardised annotations.)

The cell below downloads and assembles the atlas from GEO. The exact file names occasionally
change — if a step fails, open the GSE176078 page and check the supplementary file name.

In [ ]:
REF_H5AD = os.path.join(WORKDIR, "wu2021_brca_ref.h5ad")
CELLTYPE_KEY = "celltype_major"   # cell-type column in the reference

if not os.path.exists(REF_H5AD):
    import urllib.request, tarfile
    from scipy.io import mmread
    from scipy.sparse import csr_matrix

    url = ("https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE176078&format=file&"
           "file=GSE176078%5FWu%5Fetal%5F2021%5FBRCA%5FscRNASeq%2Etar%2Egz")
    tar_path = os.path.join(WORKDIR, "wu2021.tar.gz")
    print("downloading reference (~large, one-time)...")
    urllib.request.urlretrieve(url, tar_path)
    with tarfile.open(tar_path) as t:
        t.extractall(WORKDIR)

    # locate the four files regardless of the exact folder name
    f = lambda pat: glob.glob(os.path.join(WORKDIR, "**", pat), recursive=True)[0]
    mtx   = mmread(f("*matrix*.mtx")).tocsr()           # usually genes x cells
    genes = pd.read_csv(f("*genes*.tsv"),    header=None)[0].values
    bcs   = pd.read_csv(f("*barcodes*.tsv"), header=None)[0].values
    meta  = pd.read_csv(f("*metadata*.csv"), index_col=0)

    # orient so rows = cells
    if mtx.shape[0] == len(genes):
        mtx = mtx.T.tocsr()
    ref = sc.AnnData(X=csr_matrix(mtx), obs=pd.DataFrame(index=bcs),
                     var=pd.DataFrame(index=genes))
    ref = ref[meta.index.intersection(ref.obs_names)].copy()
    ref.obs[CELLTYPE_KEY] = meta.loc[ref.obs_names, CELLTYPE_KEY].values
    ref.write(REF_H5AD)
    print("reference assembled:", ref.shape)

ref = sc.read_h5ad(REF_H5AD)
ref.var_names_make_unique()
print(ref.obs[CELLTYPE_KEY].value_counts())

### Subsample the reference

100k cells is more than CondSCVI needs and makes training slow. We cap each cell type at a
few thousand cells — this keeps every type well represented while cutting training time.
Deconvolution quality depends on the reference being *balanced and clean*, not merely large.

In [ ]:
N_PER_TYPE = 2500
idx = (ref.obs.groupby(CELLTYPE_KEY, observed=True)
          .apply(lambda d: d.sample(min(len(d), N_PER_TYPE), random_state=0))
          .index.get_level_values(1))
ref = ref[idx].copy()
print("subsampled reference:", ref.shape)
print(ref.obs[CELLTYPE_KEY].value_counts())

## 3 · Match genes between reference and spots

Both models must use the **same genes in the same order**. We pick informative genes from
the reference (highly variable), intersect them with the genes present in the slide, and
subset both objects to that shared list — keeping raw counts intact in each.

In [ ]:
# choose HVGs on a normalised *copy* of the reference (don't disturb the raw counts)
ref_norm = ref.copy()
sc.pp.normalize_total(ref_norm, target_sum=1e4)
sc.pp.log1p(ref_norm)
sc.pp.highly_variable_genes(ref_norm, n_top_genes=2000, flavor="seurat")
hvg = ref_norm.var_names[ref_norm.var["highly_variable"]]

# shared genes = reference HVGs that also exist in the slide
shared = [g for g in hvg if g in set(st_adata.var_names)]
print(f"{len(shared)} shared genes used for deconvolution")

ref = ref[:, shared].copy()           # raw counts, shared genes
st_adata = st_adata[:, shared].copy() # raw counts, shared genes

## 4 · Train CondSCVI on the reference

CondSCVI is a conditional VAE: conditioned on the cell-type label, it learns a latent model
of each type's expression. We pass `prior="mog"` (a mixture-of-Gaussians latent prior) —
required so the next step (`DestVI.from_rna_model`) can read the learned prior correctly.

In [ ]:
CondSCVI.setup_anndata(ref, labels_key=CELLTYPE_KEY)   # uses raw counts in .X
sc_model = CondSCVI(ref, weight_obs=False, prior="mog")
sc_model.train(max_epochs=250)

sc_model.history["elbo_train"].plot(title="CondSCVI training (ELBO)"); plt.show()

## 5 · Train DestVI on the spots → proportions

DestVI takes the trained reference model and explains each spot as a mixture of the learned
cell-type signatures. After training, `get_proportions()` returns a **spots × cell-types**
table where each row sums to 1 — the cell-type composition of every spot.

In [ ]:
DestVI.setup_anndata(st_adata)               # uses raw counts in .X
st_model = DestVI.from_rna_model(st_adata, sc_model)
st_model.train(max_epochs=2500)
st_model.history["elbo_train"].iloc[10:].plot(title="DestVI training (ELBO)"); plt.show()

In [ ]:
# proportions: one column per cell type, rows sum to ~1
proportions = st_model.get_proportions()
proportions.index = st_adata.obs_names
print(proportions.head().round(3))

# store: the full table in obsm, and each cell type as its own obs column (for plotting)
st_adata.obsm["proportions"] = proportions
for ct in proportions.columns:
    st_adata.obs[ct] = proportions[ct].values

## 6 · Map cell types onto the tissue

Two views. First, the spatial distribution of a few key cell types — each spot coloured by
its estimated proportion of that type. You should see tumour and stroma/immune occupying
different regions.

In [ ]:
# pick a few biologically central types that exist in the columns
wanted = ["Cancer Epithelial", "T-cells", "CAFs", "Myeloid"]
to_plot = [c for c in wanted if c in proportions.columns][:4]
sq.pl.spatial_scatter(st_adata, color=to_plot, size=1.4, ncols=2, figsize=(6, 6))

### Cell types per spatial domain — the bridge to Notebook 3

Now connect the two notebooks: average each cell type's proportion **within each Notebook-1
domain**. This tells you what each GNN domain is *made of* — e.g. a "tumour" domain should
be dominated by Cancer Epithelial, an "immune" domain by T-cells/Myeloid. This composition
heatmap is the first real biological readout of the project.

In [ ]:
comp = (st_adata.obs.groupby("domain", observed=True)[list(proportions.columns)]
                 .mean())
plt.figure(figsize=(8, 4))
plt.imshow(comp.values, aspect="auto", cmap="viridis")
plt.xticks(range(comp.shape[1]), comp.columns, rotation=45, ha="right")
plt.yticks(range(comp.shape[0]), [f"domain {d}" for d in comp.index])
plt.colorbar(label="mean proportion"); plt.title("Cell-type composition per domain")
plt.tight_layout(); plt.show()
comp.round(3)

## 7 · Save for Notebook 3

We save the spatial object with proportions attached. Notebook 3 brings everything together:
neighbourhood / niche statistics (which cell types co-localise), the H&E morphology bridge,
and the figures that tell the project's story.

In [ ]:
out = os.path.join(WORKDIR, "brca_visium_deconvolved.h5ad")
st_adata.write(out)
print("saved ->", out)

---
### Where we are
Every spot now has both a **domain** (Notebook 1) and a **cell-type composition**
(this notebook). Worth checking before moving on: do the spatial cell-type maps line up with
the H&E image, and does the composition heatmap give each domain a sensible identity? When
you're happy, tell me and I'll build **Notebook 3 (niche analysis + morphology + figures)**.